# Análisis Exploratorio de Datos (EDA), Limpieza
**Proyecto:** Clasificador de Salud Fetal (Dataset Cardiotocography - UCI)

**Contexto:** Este notebook realiza la carga inicial de datos, inspección y preprocesamiento manual de registros de Cardiotocografía (CTG), una prueba médica para detectar sufrimiento o hipoxia fetal a tiempo.

### Objetivo
Limpiar el dataset crudo, manejar valores faltantes, prevenir la fuga de datos (*data leakage*) y realizar un análisis exploratorio para comprender la distribución y el comportamiento de las señales de la frecuencia cardíaca fetal.

---

### Descripción de las Variables (Features) por Bloque

#### 1. Línea Base y Actividad Uterina
* `LB` (*Baseline FHR*): Frecuencia cardíaca basal del feto (latidos por minuto, normalmente entre 110 y 160 lpm).
* `AC` (*Accelerations*): Número de aceleraciones de la frecuencia cardíaca por segundo (indican buena respuesta y bienestar fetal).
* `FM` (*Fetal Movements*): Número de movimientos fetales detectados por segundo.
* `UC` (*Uterine Contractions*): Número de contracciones uterinas de la madre por segundo.

#### 2. Variabilidad (Fluctuaciones de latido a latido)
* `ASTV`: Porcentaje de tiempo con variabilidad a corto plazo anormal.
* `MSTV`: Valor medio de la variabilidad a corto plazo.
* `ALTV`: Porcentaje de tiempo con variabilidad a largo plazo anormal.
* `MLTV`: Valor medio de la variabilidad a largo plazo.

#### 3. Desaceleraciones (Caídas anormales del ritmo cardíaco)
* `DL`: Desaceleraciones leves.
* `DS`: Desaceleraciones severas (altamente peligrosas).
* `DP`: Desaceleraciones prolongadas.

#### 4. Propiedades Estadísticas del Histograma Fetal
* `Width`: Ancho total del histograma de frecuencias.
* `Min` / `Max`: Valores mínimo y máximo registrados en el histograma.
* `Nmax`: Número de picos detectados en el histograma.
* `Nzeros`: Número de ceros en el histograma.
* `Mode`, `Mean`, `Median`: Estadísticos de tendencia central de los latidos (Moda, Media y Mediana).
* `Variance`: Varianza estadística de la señal.
* `Tendency`: Tendencia del histograma (-1 = asimétrico a la izquierda, 0 = simétrico, 1 = asimétrico a la derecha).

#### 5. Variable Objetivo (*Target*)
* **`NSP`** (*Fetal State Class Code*): Diagnóstico clínico final del estado fetal:
  * `1` = Normal
  * `2` = Sospechoso
  * `3` = Patológico (Riesgo de hipoxia / sufrimiento fetal)

In [2]:
#Importación de librerias

import pandas as pd # Manejo de Tablas
import numpy as np # Operaciones matemáticas 
import matplotlib.pyplot as plt # Gráficos 
import seaborn as sns # Gráficos

In [3]:
# Carga de datos

raw_data_path = '../data/raw/CTG.xls' # Ruta

df_raw = pd.read_excel(raw_data_path, sheet_name='Data', header=1) # Lee el xlsx y usa la Fila 1 como Header

df_raw = df_raw.dropna(axis=1, how='all') # Elimina columnas completamente vacias

print("Dimensiones:", df_raw.shape) # Dimensiones del DataSet
display(df_raw.head()) # Mostrar las primeras filas

Dimensiones: (2129, 42)


,b,e,AC,FM,UC,DL,DS,DP,DR,LB,...,C,D,E,AD,DE,LD,FS,SUSP,CLASS,NSP
0,240.0,357.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,120.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0,-1.0,9.0,2.0
1,5.0,632.0,4.0,0.0,4.0,2.0,0.0,0.0,0.0,132.0,...,-1.0,-1.0,-1.0,1.0,-1.0,-1.0,-1.0,-1.0,6.0,1.0
2,177.0,779.0,2.0,0.0,5.0,2.0,0.0,0.0,0.0,133.0,...,-1.0,-1.0,-1.0,1.0,-1.0,-1.0,-1.0,-1.0,6.0,1.0
3,411.0,1192.0,2.0,0.0,6.0,2.0,0.0,0.0,0.0,134.0,...,-1.0,-1.0,-1.0,1.0,-1.0,-1.0,-1.0,-1.0,6.0,1.0
4,533.0,1147.0,4.0,0.0,5.0,0.0,0.0,0.0,0.0,132.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,2.0,1.0


In [4]:
# Información del DataSet

print("--- Información General ---")
df_raw.info()

print("\n--- Valores Faltantes por Columna ---")

missing_data = df_raw.isnull().sum()
print(missing_data[missing_data > 0])

display(df_raw.tail())

--- Información General ---
<class 'pandas.DataFrame'>
RangeIndex: 2129 entries, 0 to 2128
Data columns (total 42 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   b         2126 non-null   float64
 1   e         2126 non-null   float64
 2   AC        2126 non-null   float64
 3   FM        2127 non-null   float64
 4   UC        2127 non-null   float64
 5   DL        2128 non-null   float64
 6   DS        2128 non-null   float64
 7   DP        2128 non-null   float64
 8   DR        2128 non-null   float64
 9   LB        2126 non-null   float64
 10  AC.1      2126 non-null   float64
 11  FM.1      2127 non-null   float64
 12  UC.1      2127 non-null   float64
 13  DL.1      2128 non-null   float64
 14  DS.1      2128 non-null   float64
 15  DP.1      2128 non-null   float64
 16  ASTV      2127 non-null   float64
 17  MSTV      2127 non-null   float64
 18  ALTV      2127 non-null   float64
 19  MLTV      2127 non-null   float64
 20  Width     212

,b,e,AC,FM,UC,DL,DS,DP,DR,LB,...,C,D,E,AD,DE,LD,FS,SUSP,CLASS,NSP
2124,1576.0,3049.0,1.0,0.0,9.0,0.0,0.0,0.0,0.0,140.0,...,-1.0,-1.0,1.0,-1.0,-1.0,-1.0,-1.0,-1.0,5.0,2.0
2125,2796.0,3415.0,1.0,1.0,5.0,0.0,0.0,0.0,0.0,142.0,...,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,-1.0,1.0,1.0
2126,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2127,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,NaN,...,53.0,81.0,72.0,332.0,252.0,107.0,69.0,197.0,NaN,NaN
2128,NaN,NaN,NaN,564.0,23.0,16.0,1.0,4.0,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### Hallazgos de la Inspección y Criterios de Limpieza

Tras analizar la estructura del archivo, identificamos cuatro elementos clave que debemos limpiar antes de entrenar nuestros modelos:

1. **Filas vacías o de resumen al final del archivo:** 
   Las últimas filas del documento contienen celdas vacías (`NaN`) en la etiqueta objetivo (`NSP`). No corresponden a pacientes reales, sino a cálculos automáticos que Excel genera al final de la tabla, por lo que deben eliminarse.

2. **Corrección del sesgo por duración del examen (Tasas vs. Conteos):** 
   El dataset incluye dos versiones de los eventos (como aceleraciones y contracciones):
   * *Conteos absolutos* (`AC`, `FM`, `UC`, etc.): Suman el total de veces que ocurrió el evento. Esto genera un sesgo, ya que un feto monitoreado durante más tiempo tendrá naturalmente más eventos que uno monitoreado por pocos minutos.

   * *Tasas normalizadas* (`AC.1`, `FM.1`, etc.): Dividen esos eventos entre el tiempo exacto de duración de la prueba. Por lo tanto, **eliminamos los conteos brutos y nos quedamos únicamente con las tasas normalizadas por segundo**.

3. **Prevención de Fuga de Datos (*Data Leakage*):** 
   Las columnas de patrones morfológicos (`A` hasta `SUSP`) y la clase general (`CLASS`) representan el criterio de diagnóstico intermedio que utilizaron los médicos. Si las dejamos en el dataset, el modelo "haría trampa" porque estas variables revelan casi por completo el resultado de la variable objetivo (`NSP`). Deben ser eliminadas.

4. **Eliminación de metadatos clínicos irrelevantes:** 
   Variables como el segundo de inicio (`b`) y el segundo de fin (`e`) solo sirvieron para calcular las tasas temporales y no aportan ningún valor predictivo sobre la salud del feto.

In [5]:
# Eliminación de Datos 

# Elimina filas que tengan valores faltantes en NSP
df_clean = df_raw.dropna(subset=['NSP']).copy() 

# Columnas de apoyo en medición y conteos absolutos
cols_metadata_and_counts = ['b', 'e', 'DR', 'AC', 'FM', 'UC', 'DL', 'DS', 'DP'] 

# Clasifiaciones realizadas por expertos que pueden generar data leakages
cols_leakage = ['A', 'B', 'C', 'D', 'E', 'AD', 'DE', 'LD', 'FS', 'SUSP', 'CLASS'] 

# DataSet limpio
df_clean = df_clean.drop(columns=cols_metadata_and_counts + cols_leakage)

# Renombrar columnas para elimiar el .1
rename_dict = {
    'AC.1': 'AC', 'FM.1': 'FM', 'UC.1': 'UC', 
    'DL.1': 'DL', 'DS.1': 'DS', 'DP.1': 'DP'
}
df_clean = df_clean.rename(columns=rename_dict)

print(f"Dimensiones limpias y listas para análisis: {df_clean.shape}")
print("\nColumnas predictivas finales (Features + Target):")
print(df_clean.columns.tolist())

Dimensiones limpias y listas para análisis: (2126, 22)

Columnas predictivas finales (Features + Target):
['LB', 'AC', 'FM', 'UC', 'DL', 'DS', 'DP', 'ASTV', 'MSTV', 'ALTV', 'MLTV', 'Width', 'Min', 'Max', 'Nmax', 'Nzeros', 'Mode', 'Mean', 'Median', 'Variance', 'Tendency', 'NSP']


In [6]:
# Verificación final de integridad

print("--- Verificación de Valores Nulos Post-Limpieza ---")
print(df_clean.isnull().sum().sum(), "valores nulos encontrados en total.")
print(f"Dimensiones definitivas: {df_clean.shape}")

--- Verificación de Valores Nulos Post-Limpieza ---
0 valores nulos encontrados en total.
Dimensiones definitivas: (2126, 22)


In [15]:
# Guardar el csv del dataset limpio 

import os

os.makedirs('../data/processed', exist_ok=True)
df_clean.to_csv('../data/processed/CTG_clean.csv', index=False)

print(f"Dimensiones del archivo guardado: {df_clean.shape}")

Dimensiones del archivo guardado: (2126, 22)
